<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

## Newton's Method and Stopping

**Use curvature for the update and separate a stopping test from proof of a minimum.**

Keep both clock settings unrestricted, the observations, and the squared-error objective. Calibration and the correction bound remain removed.

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 1 · From slope to step

The fixed errors $d_i$ are $-2,-1,-1,0$ min at times $t_i=0,1,2,3$ h, respectively, for $i=1,\ldots,4$. Choose initial correction $q$ (min) and rate correction $r$ (min/h). The remaining errors and squared-error objective are

$$
y_i=d_i+q+rt_i,\qquad \phi(q,r)=\sum_{i=1}^4 y_i^2.
$$

Equations below use numerical values in these stated units.

The observations and squared-error objective are retained. The calibration equality and correction bound are removed: the updates are unconstrained. Write $x=(q,r)$ and $f(x)=\phi(q,r)$.

With $r=0$, the slope is $f'(q)=8(q-1)$ and the curvature is $f''(q)=8$. A step $p$ that makes the locally linearized slope zero satisfies $f'(q)+f''(q)p=0$, giving $p=-f'(q)/f''(q)$. From $q=0$, this is $p=1$.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/04-2_iterative_optimization_gradient_descent_and_newton_method/assets/newton_slope.svg" alt="Dividing the offset slope minus 8 by curvature 8 gives a Newton change of plus 1." width="520" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 2 · Use a linear system for two settings

The **Hessian** $B$ collects the second partial derivatives. For this objective,

$$
\nabla f=\begin{bmatrix}8q+12r-8\\12q+28r-6\end{bmatrix},
\qquad B=\begin{bmatrix}8&12\\12&28\end{bmatrix}.
$$

Linearizing the gradient gives $\nabla f(x+p)\approx\nabla f(x)+Bp$. Newton's direction solves

$$
Bp=-\nabla f(x),\qquad x_{\mathrm{new}}=x+\eta p.
$$

The damping hyperparameter $0<\eta\le1$ controls how much of the direction is used. Solve the system without explicitly forming $B^{-1}$.

At $x=(0,0)$, $Bp=(8,6)^{\mathsf T}$ gives $p=(1.9,-0.6)$. A full step ($\eta=1$) reaches the minimum with score $0.2$ min². A half step gives $(0.95,-0.3)$ with score $1.65$ min².

Compare directions from the same start: Newton corrects for curvature coupling, so even its initial rate direction differs from gradient descent.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/04-2_iterative_optimization_gradient_descent_and_newton_method/assets/clock_update_paths.svg" alt="A full Newton step reaches the quadratic minimum directly while gradient descent takes several updates." width="520" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

One-step convergence is special to this quadratic with a constant positive-definite Hessian. In general, the Hessian changes; it may be singular or give a direction that fails to decrease the objective. Fewer updates alone do not imply lower runtime because each Newton step requires a linear solve.

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 3 · Check why a run stopped

Let $\varepsilon>0$ be a gradient tolerance and $K$ a maximum update count. Gradient descent uses $x_{\mathrm{new}}=x-0.05\nabla f(x)$; damped Newton uses $x_{\mathrm{new}}=x+\eta p$. Both start at $(0,0)$.

| Observation | Meaning |
|:---|:---|
| $\|\nabla f\|_2\le\varepsilon$ | Approximate stationarity in the stated coordinates |
| A tiny update | Could be caused by a tiny step-size hyperparameter |
| $K$ updates reached | Work limit exhausted; inspect the gradient residual |

For example, a gradient step with $\alpha=10^{-12}$ has length $10^{-11}$ at $(0,0)$, while the gradient norm remains about $10$. This is not evidence of convergence.

The global guarantee is separate: for nonzero $p=(p_q,p_r)$,

$$
p^{\mathsf T}Bp=2\sum_{i=1}^4(p_q+t_ip_r)^2>0.
$$

Hence this quadratic is strictly convex and its stationary point $(1.9,-0.6)$ is the unique global minimum. A short trace alone would not prove that. Compare the stopping reasons while keeping the problem and initial decision fixed.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/04-2_iterative_optimization_gradient_descent_and_newton_method/assets/stopping_conditions.svg" alt="A satisfied gradient tolerance, a tiny step, and an update limit are different stopping reasons." width="520" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 4 · Interactive comparison

Both methods start at $(0,0)$ on the same objective. The sliders change Newton damping $\eta$, gradient tolerance $\varepsilon=10^a$, and the update limit $K$. Gradient descent keeps $\alpha=0.05$. Each run stops at its first satisfied gradient test or at the limit; a budget stop is labeled explicitly. Norms below $10^{-14}$ are placed at that plotting floor only.

In [ ]:
import sys
import numpy as np
import matplotlib

if sys.platform != "emscripten":
    try:
        matplotlib.use("widget", force=True)
    except (RuntimeError, ValueError):
        from matplotlib.backends import backend_registry
        backend_registry._clear()
        matplotlib.use("widget", force=True)
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider

BLUE, TEAL, ORANGE, PURPLE, GRAY = "#2878b5", "#168578", "#e78b24", "#8854a5", "#707070"
plt.rcParams.update({"font.size": 11, "axes.titlesize": 11})

# Fixed observations and the physical response.
observed_errors = np.array([-2., -1., -1., 0.])  # min
observation_times = np.arange(4.)  # h

def evaluate_clock(decision):
    correction, rate = decision
    errors = observed_errors + correction + rate * observation_times
    gradient = 2 * np.array([errors.sum(), observation_times @ errors])
    return {"decision": np.array(decision), "errors": errors,
            "objective": float(errors @ errors), "gradient": gradient}

hessian = np.array([[8., 12.], [12., 28.]])

def run_method(method, damping, tolerance, max_steps):
    decision = np.zeros(2)
    history = []
    for step in range(max_steps+1):
        result = evaluate_clock(decision)
        norm = float(np.linalg.norm(result["gradient"]))
        history.append(norm)
        if norm <= tolerance:
            reason = "gradient tolerance"
            break
        if step == max_steps:
            reason = "update limit"
            break
        direction = (-0.05*result["gradient"] if method == "GD" else
                     damping*np.linalg.solve(hessian, -result["gradient"]))
        decision = decision + direction
    return {"norms": np.array(history), "reason": reason,
            "steps": step, "result": result}

In [ ]:
figure, axis = plt.subplots(figsize=(5.6, 5.5))
figure.subplots_adjust(left=0.16, right=0.96, top=0.91, bottom=0.45)
gd_line, = axis.plot([], [], color=BLUE, label="Gradient descent")
newton_line, = axis.plot([], [], "o-", color=ORANGE, markersize=4, label="Newton")
tolerance_line = axis.axhline(1e-6, color=TEAL, linestyle="--")
axis.set(xlabel="Update k", ylabel="Gradient norm (log scale)", yscale="log",
         title="Tolerance and work limit give different stopping reasons")
axis.grid(alpha=0.25)
axis.legend(fontsize=10)
status = figure.text(0.16, 0.30, "", va="top")
damping_slider = Slider(figure.add_axes([0.25, 0.18, 0.58, 0.028]), "eta",
                        0.05, 1, valinit=1, valstep=0.05, color=PURPLE)
tolerance_slider = Slider(figure.add_axes([0.25, 0.115, 0.58, 0.028]), "log10 eps",
                          -8, -1, valinit=-6, valstep=1, valfmt="%d", color=PURPLE)
budget_slider = Slider(figure.add_axes([0.25, 0.05, 0.58, 0.028]), "K",
                       1, 200, valinit=150, valstep=1, valfmt="%d", color=PURPLE)

def update(_):
    tolerance, budget = 10.**tolerance_slider.val, int(budget_slider.val)
    runs = [run_method(method, damping_slider.val, tolerance, budget)
            for method in ("GD", "Newton")]
    for line, run in zip((gd_line, newton_line), runs):
        line.set_data(range(len(run["norms"])), np.maximum(run["norms"], 1e-14))
    tolerance_line.set_ydata([tolerance, tolerance])
    axis.set_xlim(-0.5, max(run["steps"] for run in runs)+1)
    axis.set_ylim(min(tolerance/10, max(1e-14, min(r["norms"].min() for r in runs))/3), 30)
    status.set_text("\n".join(f"{name}: {run['steps']} updates; {run['reason']}"
                             for name, run in zip(("GD", "Newton"), runs)))
    figure.canvas.draw_idle()

figure._sliders = [damping_slider, tolerance_slider, budget_slider]
for slider in figure._sliders:
    slider.on_changed(update)
update(None)
plt.show()